<a href="https://colab.research.google.com/github/McChandra/medical_chatbot/blob/main/responsive-healthbrowse-auth/healthbrowse/inference/HealthBrowse_Llama_QLoRA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HealthBrowse: Llama 3 + QLoRA on free Colab

Train an adapter for **General** answers, six source-cited decks and buddy follow-ups. This notebook runs downloaded model weights; it makes **no paid inference API calls**. Hugging Face is used to download the base weights, subject to Meta's access terms.

**Status:** prepared workflow; no training results or medically validated adapter are supplied. Free Colab GPUs and session duration are not guaranteed. This is an interactive research notebook, not a public web server or a replacement for hosting HealthBrowse.

Before starting:
1. Choose **Runtime → Change runtime type → GPU**. A T4 may be assigned, but is not guaranteed.
2. Obtain authorized access to [Meta-Llama-3-8B-Instruct](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct).
3. Add a Hugging Face **read** token to Colab Secrets as `HF_TOKEN`; grant this notebook access. Do not put a token in a code cell. Paid Inference Providers permission is unnecessary for weight downloads.
4. Prepare `train.jsonl` and `validation.jsonl` in the source-grounded format in `colab-qlora.md`. Keep diseases and publisher pages separate between splits. Existing QA-only CSV rows require conversion and source review first. No patient records or visitor chats.
5. Keep several GB of Drive space available for adapters, optimizer checkpoints and reports. Downloaded base weights stay on the temporary VM, not Drive.

Run cells in order. First use the short 50-step experiment; it tests the pipeline and does not establish answer quality.

## 1. Get the project code
Use the PR branch until it is merged. This cell records its exact commit, or a training-source fingerprint for an uploaded ZIP. On a resumed run, the code version must match the saved experiment configuration.
If this repository is private, upload an authorized source ZIP using Colab's Files panel and set `REPO` to its extracted directory instead; never put a GitHub token in a clone URL.

In [ ]:
from pathlib import Path
import subprocess, sys, json, os
REPO = Path('/content/medical_chatbot')
SOURCE_REF = 'feature/responsive-healthbrowse-auth'
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch',SOURCE_REF,
                    'https://github.com/McChandra/medical_chatbot.git',str(REPO)],check=True)
INFERENCE = REPO / 'healthbrowse' / 'inference'
assert (INFERENCE/'train_adapter.py').is_file(), 'Choose the checkout containing the new inference files.'
if (REPO/'.git').exists():
    SOURCE_VERSION = subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Use an unmodified checkout.'
else:
    import hashlib
    digest=hashlib.sha256()
    for source in sorted([*INFERENCE.glob('*.py'),INFERENCE/'requirements.txt']):
        digest.update(source.name.encode()+b'\0'+source.read_bytes())
    SOURCE_VERSION='archive-sha256:'+digest.hexdigest()
print('Source version:', SOURCE_VERSION)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(INFERENCE/'requirements.txt')],check=True)

Source version: b287629223cfc5f916dee75c2d4321d0243a3277


## 2. Check the assigned GPU
The script chooses FP16 for T4-class GPUs and native BF16 where supported. The 2,048-token, rank-8 profile is a starting configuration, not a guarantee that every session will fit. If no GPU is available, stop and try an available authorized session later. Do not use keep-alive scripts or multiple accounts to bypass limits.

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU assigned. Select a GPU runtime; availability depends on Colab.')
sys.path.insert(0,str(INFERENCE))
from train_adapter import precision_flags
props = torch.cuda.get_device_properties(0)
FLAGS = precision_flags(torch.cuda.get_device_capability()[0],torch.cuda.is_bf16_supported())
DTYPE = torch.bfloat16 if FLAGS['bf16'] else torch.float16
print({'GPU':props.name,'VRAM_GiB':round(props.total_memory/2**30,1),'dtype':str(DTYPE)})

## 3. Access the weights and save the experiment configuration
Mount only your own research Drive. Checkpoints remain there when the VM ends. This notebook does not upload weights publicly. To start a different experiment, use a **new RUN_NAME** instead of overwriting an old run.

In [ ]:
from google.colab import userdata, drive
from huggingface_hub import HfApi
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
assert os.environ['HF_TOKEN'], 'Add the read token to Colab Secrets, not notebook source.'
drive.mount('/content/drive')
RUN_NAME = 'llama3-general-qlora-001'
RUN_DIR = Path('/content/drive/MyDrive/HealthBrowse') / RUN_NAME
RUN_DIR.mkdir(parents=True,exist_ok=True)
MODEL_ID = 'meta-llama/Meta-Llama-3-8B-Instruct'
MAX_LENGTH = 2048
MAX_STEPS = 50  # Pipeline experiment; use a new run with -1 for epoch-based training.
EPOCHS = 2
SAVE_STEPS = 10
LORA_RANK = 8
config_path = RUN_DIR / 'experiment.json'
previous = json.loads(config_path.read_text()) if config_path.exists() else None
BASE_REVISION = previous['base_revision'] if previous else HfApi().model_info(MODEL_ID).sha
config = {'source_version':SOURCE_VERSION,'model':MODEL_ID,'base_revision':BASE_REVISION,
          'max_length':MAX_LENGTH,'max_steps':MAX_STEPS,'epochs':EPOCHS,
          'save_steps':SAVE_STEPS,'lora_rank':LORA_RANK}
if previous and previous != config:
    raise ValueError('Run settings/source differ. Restore the recorded source/settings or choose a new RUN_NAME.')
config_path.write_text(json.dumps(config,indent=2))
print(config)  # No credentials in this configuration.

## 4. Supply training and validation examples
Upload exactly `train.jsonl` and `validation.jsonl`. Only use public source-grounded examples with recorded data-license and clinical review. Do not mark unreviewed rows as reviewed just to pass validation. The guide includes the exact schema and explains how MedQuAD/MedQA-style records differ from these training targets.

If you only have raw QA files, stop here and prepare the source-grounded training set first. This notebook does not invent citation labels or claim clinical review.

In [ ]:
from google.colab import files
DATA_DIR = RUN_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)
TRAIN = DATA_DIR/'train.jsonl'
VALIDATION = DATA_DIR/'validation.jsonl'
if not (TRAIN.exists() and VALIDATION.exists()):
    uploaded = files.upload()
    if set(uploaded) != {'train.jsonl','validation.jsonl'}:
        raise ValueError('Select exactly train.jsonl and validation.jsonl.')
    for name, data in uploaded.items():
        target=DATA_DIR/name
        if target.exists() and target.read_bytes()!=data:
            raise ValueError('Existing data differs. Choose a new run instead of replacing it.')
        target.write_bytes(data)
    del uploaded
from train_adapter import load_examples, validate_split
train_rows, validation_rows = load_examples(TRAIN),load_examples(VALIDATION)
validate_split(train_rows,validation_rows)
print({'training_examples':len(train_rows),'validation_examples':len(validation_rows),'split_check':'passed'})

## 5. Check token lengths and recover any interrupted checkpoint
Overlength examples fail explicitly. Shorten evidence thoughtfully, keep complete supporting passages, or select a new experiment with an appropriate length. Do not silently truncate clinical qualifiers or target citations.

Only resume your own checkpoints. A new run, changed data, or changed settings require a new output directory. If interrupted before the first complete checkpoint, keep the old run and begin a new one.

In [ ]:
from transformers import AutoTokenizer
from transformers.trainer_utils import get_last_checkpoint
from model_settings import validate_base
validate_base(MODEL_ID,BASE_REVISION,MAX_LENGTH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID,revision=BASE_REVISION,trust_remote_code=False)
lengths = [len(tokenizer.apply_chat_template(row['messages'],tokenize=True))
           for row in train_rows+validation_rows]
print({'longest_example_tokens':max(lengths),'limit':MAX_LENGTH})
if max(lengths)>MAX_LENGTH:
    raise ValueError('Overlength examples: revise the prepared data explicitly in a new run.')
OUTPUT = RUN_DIR/'adapter'
checkpoint = get_last_checkpoint(str(OUTPUT)) if OUTPUT.exists() else None
cmd = [sys.executable,str(INFERENCE/'train_adapter.py'),'--train',str(TRAIN),'--validation',str(VALIDATION),
       '--model',MODEL_ID,'--revision',BASE_REVISION,'--method','qlora','--output',str(OUTPUT),
       '--max-length',str(MAX_LENGTH),'--max-steps',str(MAX_STEPS),'--epochs',str(EPOCHS),
       '--save-steps',str(SAVE_STEPS),'--lora-rank',str(LORA_RANK)]
if checkpoint:
    cmd += ['--resume-from-checkpoint',checkpoint]
    print('Resuming:',Path(checkpoint).name)
subprocess.run(cmd+['--validate-only'],check=True)

## 6. Train the adapter
Only adapter parameters are updated; the 4-bit base stays frozen. Loss is computed on assistant targets only. Checkpoints save every configured number of optimizer steps, with the latest two retained. Work since the last completed checkpoint can be lost on interruption. Resume by reopening this notebook and rerunning cells 1–6 with the original settings.

This runs a real training job on your assigned GPU; duration and memory use depend on your data and GPU. A 50-step run is a smoke experiment, not a completed medical model.

In [ ]:
subprocess.run(cmd,check=True)
assert (OUTPUT/'adapter_model.safetensors').is_file(), 'Training did not produce a final adapter.'
print('Saved adapter:',OUTPUT)
print(json.loads((OUTPUT/'eval_results.json').read_text()))

## 7. Compare the base model and adapter on held-out questions
Use identical prompts and greedy generation. This samples at most five held-out validation rows. It is a sanity check, not a statistically adequate benchmark. Keep a separate untouched test split for the final research study.

The check below verifies parseability, deck IDs, source IDs and exact quote membership. A real quote may still fail to support a generated claim; manually assess entailment, completeness, readability and dangerous omissions. No score here certifies clinical safety.

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from model_settings import validate_adapter
validate_adapter(OUTPUT,MODEL_ID,BASE_REVISION)
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
                        bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=DTYPE)
base=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=BASE_REVISION,
    quantization_config=quant,torch_dtype=DTYPE,device_map={'':0},trust_remote_code=False)
model=PeftModel.from_pretrained(base,str(OUTPUT))
model.eval()
tokenizer.pad_token=tokenizer.eos_token

def generate(messages):
    inputs=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,
                                        return_tensors='pt',return_dict=True).to(model.device)
    count=inputs['input_ids'].shape[-1]
    budget=min(1024,model.config.max_position_embeddings-count)
    if budget<=0:
        raise ValueError('Prompt exceeds model context.')
    with torch.inference_mode():
        output=model.generate(**inputs,max_new_tokens=budget,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    tokens=output[0,count:]
    return tokenizer.decode(tokens,skip_special_tokens=True),len(tokens)>=budget

def structure_check(text,payload,truncated):
    if truncated:
        return {'passed':False,'reason':'output token limit reached'}
    try:
        data=json.loads(text)
        expected=[1,2,3,4,5,6] if payload['mode']=='search' else []
        if sorted(d['id'] for d in data['decks'])!=expected:
            raise ValueError('deck IDs')
        evidence={s['id']:s['text'] for s in payload['evidence']}
        claims=[c for p in [data['answer'],*data['decks']] for c in p['claims']]
        for claim in claims:
            if not claim['text'].strip() or not claim['support']:
                raise ValueError('missing text/support')
            for support in claim['support']:
                quote=support['quote']
                if len(quote.strip())<20 or quote not in evidence[support['id']]:
                    raise ValueError('unsupported quote')
        return {'passed':True,'claims':len(claims),'entailment_review':'not performed'}
    except (ValueError,KeyError,TypeError) as e:
        return {'passed':False,'reason':str(e)}

comparisons=[]
for row in validation_rows[:5]:
    messages=row['messages'][:-1]
    payload=json.loads(messages[1]['content'])
    with model.disable_adapter():
        before,before_cut=generate(messages)
    after,after_cut=generate(messages)
    item={'topic':row['topic'],'question':payload.get('question'),
          'base':before,'adapter':after,
          'base_structure':structure_check(before,payload,before_cut),
          'adapter_structure':structure_check(after,payload,after_cut)}
    comparisons.append(item)
    print(json.dumps(item,indent=2))
(RUN_DIR/'comparison.json').write_text(json.dumps(comparisons,indent=2))
print('Saved held-out comparison; clinical review is still required.')

## 8. Download the adapter and reports
This exports the adapter and tokenizer files, not the large base model or optimizer checkpoints. The complete checkpoints remain in Drive for recovery. Treat the exported adapter as research-only until evaluated; do not redistribute the underlying datasets without permission.

In [ ]:
import zipfile
archive=RUN_DIR/'Llama-3-HealthBrowse-General-QLoRA.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT.iterdir():
        if path.is_file() and path.suffix in {'.json','.safetensors','.model','.txt'}:
            z.write(path,'adapter/'+path.name)
    z.write(config_path,'experiment.json')
    if (RUN_DIR/'comparison.json').exists():
        z.write(RUN_DIR/'comparison.json','comparison.json')
files.download(str(archive))

## 9. Connect HealthBrowse after evaluation
This notebook does **not** activate AI on the public Site. Run the base and adapter on your own or authorized shared compute, behind an authenticated HTTPS endpoint. Keep HealthBrowse's retrieval, source checks and review pipeline; fine-tuning is not a substitute for those controls.

On that host, use the existing `inference/serve.py` with `MODEL_ID`, the pinned `MODEL_REVISION`, `ADAPTER_PATH` and a server-only `INFERENCE_API_KEY`. The current serving script loads full-precision weights, so its memory requirements are different from this notebook's 4-bit evaluation loader. Do not assume it fits the same free Colab GPU.

The Site then needs `MODEL_GENERAL_URL`, `MODEL_GENERAL_ID`, `MODEL_GENERAL_KEY` and `MODEL_GENERAL_CONTEXT_TOKENS=8192`, followed by a deployment and an authenticated end-to-end test. These are self-hosted endpoint credentials, not Featherless credentials.

Free Colab is not an always-on backend. Do not tunnel the public website through a free managed notebook or use keep-alive workarounds.

### Sources
- [QLoRA paper](https://arxiv.org/abs/2305.14314)
- [PEFT quantization guide](https://huggingface.co/docs/peft/v0.17.0/developer_guides/quantization)
- [Trainer checkpoints](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer)
- [Meta Llama 3 model and terms](https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct)
- [Colab FAQ and restrictions](https://research.google.com/colaboratory/faq.html)

Built with Meta Llama 3. No clinical certification, guaranteed factuality, or completed training result is claimed.